# 🖼️ DETECTAI — AI Image Detector Fine-Tuning
**Platform:** Google Colab (T4/A100 GPU)  
**Output model:** `saghi776/aiscern-image-detector`  
**Base model:** `microsoft/swin-base-patch4-window7-224`  
**Detects:** Midjourney v5/v6 · SDXL / Flux · DALL-E 3 · Gemini Nano / Imagen · Grok Aurora · Adobe Firefly · Leonardo AI · Real human faces  
**Target:** F1 ≥ 0.90 · Accuracy ≥ 0.90  

> **Website wire-up:** After training, set `MODELS.image_primary = 'saghi776/aiscern-image-detector'` in `frontend/lib/inference/hf-analyze.ts`


In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────
!pip install -q transformers==4.40.0 datasets accelerate timm evaluate \
    scikit-learn pillow huggingface_hub deepface opencv-python-headless \
    torch torchvision scipy albumentations matplotlib seaborn tqdm

In [ ]:
# ── CELL 2: Authenticate HuggingFace ─────────────────────────
from google.colab import userdata
from huggingface_hub import login
import os

HF_TOKEN = userdata.get('HF_TOKEN')  # Add secret HF_TOKEN in Colab Secrets panel
login(token=HF_TOKEN)
print("✅ Logged in to HuggingFace")

HF_REPO = "saghi776/aiscern-image-detector"
print(f"🎯 Target model repo: {HF_REPO}")

In [ ]:
# ── CELL 3: GPU check ────────────────────────────────────────
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# ── CELL 4: Load multi-source datasets ───────────────────────
# Sources: DiffusionDB (AI), CIFAKE (AI+Real), Flickr30k (Real),
#          OpenImages subset, FaceForensics frames, real portrait data
from datasets import load_dataset, concatenate_datasets, Dataset
import pandas as pd
from PIL import Image
import requests
from io import BytesIO
import numpy as np

print("Loading datasets — this may take 10-15 minutes on first run...")

# --- AI IMAGE DATASETS ---
# 1. DiffusionDB (Stable Diffusion) — massive AI image corpus
print("\n[1/6] Loading DiffusionDB (SD generated images)...")
diffdb = load_dataset(
    "poloclub/diffusiondb",
    "2m_random_1k",          # start with 1k subset, increase to 'large_random_100k' for production
    split="train",
    trust_remote_code=True
)
print(f"  DiffusionDB: {len(diffdb)} rows")

# 2. CIFAKE — synthetic vs real images benchmark
print("\n[2/6] Loading CIFAKE (AI vs Real benchmark)...")
cifake = load_dataset("judegrant/cifake", split="train", trust_remote_code=True)
print(f"  CIFAKE: {len(cifake)} rows")

# 3. Detectai platform's own accumulated dataset
print("\n[3/6] Loading platform dataset (saghi776/detectai-dataset)...")
try:
    platform_ds = load_dataset(
        "saghi776/detectai-dataset",
        name="image_en",
        split="train",
        trust_remote_code=True
    )
    print(f"  Platform image dataset: {len(platform_ds)} rows")
    has_platform = len(platform_ds) > 100
except Exception as e:
    print(f"  Platform dataset not ready yet ({e}) — skipping")
    has_platform = False

# 4. AI-generated faces dataset
print("\n[4/6] Loading AI-face datasets...")
try:
    ai_faces = load_dataset("tkarras/celeba_hq_demo", split="train", trust_remote_code=True)
    print(f"  CelebA-HQ (real faces): {len(ai_faces)} rows")
except:
    ai_faces = None
    print("  Skipping CelebA-HQ")

# 5. Fake faces from StyleGAN
print("\n[5/6] Loading StyleGAN fake faces...")
try:
    fake_faces = load_dataset("Warvito/diffusion-cats", split="train", trust_remote_code=True)
    print("  Loaded supplementary AI images")
except:
    fake_faces = None

# 6. Real photos benchmark
print("\n[6/6] Loading Flickr real images...")
try:
    real_photos = load_dataset("nlphuji/flickr30k", split="test[:2000]", trust_remote_code=True)
    print(f"  Flickr30k real subset: {len(real_photos)} rows")
except:
    real_photos = None

print("\n✅ Dataset loading complete")

In [ ]:
# ── CELL 5: Build unified labeled dataframe ──────────────────
import pandas as pd
from PIL import Image
import numpy as np

records = []

# From DiffusionDB — all AI-generated
for item in diffdb:
    try:
        img = item.get('image') or item.get('img')
        if img and hasattr(img, 'size'):
            records.append({'image': img, 'label': 1})  # 1 = AI
    except: pass

# From CIFAKE
label_col = 'label' if 'label' in cifake.column_names else 'labels'
for item in cifake:
    try:
        img = item.get('image') or item.get('img')
        lbl = item.get(label_col, 0)
        if img and hasattr(img, 'size'):
            records.append({'image': img, 'label': int(lbl)})
    except: pass

# From platform dataset if available
if has_platform:
    for item in platform_ds:
        try:
            lbl = 1 if item.get('label','human') == 'ai' else 0
            img = item.get('image') or item.get('img')
            if img and hasattr(img, 'size') and item.get('quality', 1.0) >= 0.5:
                records.append({'image': img, 'label': lbl})
        except: pass

print(f"Total samples: {len(records)}")
ai_count   = sum(1 for r in records if r['label']==1)
real_count = sum(1 for r in records if r['label']==0)
print(f"  AI images:   {ai_count}")
print(f"  Real images: {real_count}")
print(f"  Balance:     {ai_count/(ai_count+real_count)*100:.1f}% AI")

# Shuffle
import random
random.shuffle(records)

# Train/val split (90/10)
split = int(len(records) * 0.9)
train_records = records[:split]
val_records   = records[split:]
print(f"\nTrain: {len(train_records)} | Val: {len(val_records)}")

In [ ]:
# ── CELL 6: Advanced augmentation + face analysis pipeline ──
from torchvision import transforms
import torchvision.transforms.functional as TF
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
from PIL import Image
import torch

# ViT/Swin input size
IMG_SIZE = 224

# --- Augmentation ---
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05, p=0.5),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.3),
    A.RandomRotate90(p=0.1),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3,5), p=1.0),
        A.Sharpen(alpha=(0.1,0.3), p=1.0),
    ], p=0.3),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

def extract_face_features(img_np):
    """Extract face quality signals: texture, expression cues, symmetry."""
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    # Laplacian variance (sharpness / texture richness)
    lap_var  = cv2.Laplacian(gray, cv2.CV_64F).var()
    # Local Binary Pattern texture entropy (AI faces are smoother)
    mean_px  = float(gray.mean())
    std_px   = float(gray.std())
    # Edge density — AI generators over-smooth fine details
    edges    = cv2.Canny(gray, 100, 200)
    edge_density = float(edges.mean())
    # Frequency analysis — GAN/diffusion artifacts in high frequency
    f_transform  = np.fft.fft2(gray.astype(np.float32))
    f_shift      = np.fft.fftshift(f_transform)
    mag_spectrum = np.abs(f_shift)
    h, w = gray.shape
    cy, cx = h//2, w//2
    r1, r2 = min(h,w)//8, min(h,w)//4
    def ring(r):
        Y, X = np.ogrid[:h, :w]
        d = np.sqrt((X-cx)**2 + (Y-cy)**2)
        return mag_spectrum[(d>=r) & (d<r+5)].mean() if len(mag_spectrum[(d>=r)&(d<r+5)])>0 else 0
    freq_ratio = ring(r2) / (ring(r1) + 1e-6)
    return [lap_var/1000, mean_px/255, std_px/100, edge_density/50, freq_ratio]

class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, records, transform, use_face_features=True):
        self.records = records
        self.transform = transform
        self.use_face = use_face_features

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img = rec['image']
        label = rec['label']

        # Convert to numpy RGB
        if not isinstance(img, np.ndarray):
            img = np.array(img.convert('RGB'))
        if img.shape[-1] == 4:
            img = img[:,:,:3]

        transformed = self.transform(image=img)
        tensor = transformed['image']
        return tensor, torch.tensor(label, dtype=torch.long)

train_ds = ImageDataset(train_records, train_transform)
val_ds   = ImageDataset(val_records,   val_transform)
print(f"Train dataset: {len(train_ds)} samples")
print(f"Val dataset:   {len(val_ds)} samples")

In [ ]:
# ── CELL 7: Load Swin Transformer model ─────────────────────
from transformers import SwinForImageClassification, SwinConfig
import torch
import torch.nn as nn

MODEL_NAME = "microsoft/swin-base-patch4-window7-224"
print(f"Loading base model: {MODEL_NAME}")

model = SwinForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "human-real", 1: "ai-generated"},
    label2id={"human-real": 0, "ai-generated": 1},
    ignore_mismatched_sizes=True
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Model loaded on:  {device}")

In [ ]:
# ── CELL 8: Training configuration ──────────────────────────
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

BATCH_SIZE  = 32   # reduce to 16 if OOM
EPOCHS      = 5
LR          = 2e-5
WEIGHT_DECAY= 0.01

# Class-balanced loss (handles AI/real imbalance)
labels_array = np.array([r['label'] for r in train_records])
cw = compute_class_weight('balanced', classes=np.unique(labels_array), y=labels_array)
class_weights = torch.tensor(cw, dtype=torch.float32).to(device)
print(f"Class weights — Real: {cw[0]:.3f} | AI: {cw[1]:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"Batches/epoch: {len(train_loader)}")
print(f"Training for {EPOCHS} epochs")

In [ ]:
# ── CELL 9: Training loop ────────────────────────────────────
from sklearn.metrics import f1_score, accuracy_score, classification_report
import time

best_f1 = 0.0
best_model_state = None

for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    train_loss, train_preds, train_labels_all = 0, [], []
    t0 = time.time()

    for batch_idx, (imgs, labels) in enumerate(train_loader):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(pixel_values=imgs)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.logits.argmax(-1).cpu().numpy()
        train_preds.extend(preds)
        train_labels_all.extend(labels.cpu().numpy())

        if (batch_idx+1) % 20 == 0:
            print(f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}", end="\r")

    # ── Validate ──
    model.eval()
    val_preds, val_labels_all = [], []
    val_loss = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(pixel_values=imgs)
            loss = criterion(outputs.logits, labels)
            val_loss += loss.item()
            preds = outputs.logits.argmax(-1).cpu().numpy()
            val_preds.extend(preds)
            val_labels_all.extend(labels.cpu().numpy())

    scheduler.step()

    train_f1  = f1_score(train_labels_all, train_preds, average='weighted')
    val_f1    = f1_score(val_labels_all, val_preds, average='weighted')
    val_acc   = accuracy_score(val_labels_all, val_preds)
    elapsed   = time.time() - t0

    print(f"\nEpoch {epoch+1}/{EPOCHS} ({elapsed:.0f}s)")
    print(f"  Train Loss: {train_loss/len(train_loader):.4f} | F1: {train_f1:.4f}")
    print(f"  Val   Loss: {val_loss/len(val_loader):.4f}   | F1: {val_f1:.4f} | Acc: {val_acc:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"  ✅ New best F1: {best_f1:.4f} — saved")

print(f"\n🏆 Best Val F1: {best_f1:.4f}")

In [ ]:
# ── CELL 10: Final evaluation + classification report ─────
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load best weights
model.load_state_dict(best_model_state)
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(pixel_values=imgs)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds = probs.argmax(-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs[:,1].cpu().numpy())

print("="*55)
print("FINAL EVALUATION — AI Image Detector")
print("="*55)
print(classification_report(all_labels, all_preds,
      target_names=["Real/Human", "AI-Generated"], digits=4))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Real","AI"], yticklabels=["Real","AI"])
plt.title(f'Confusion Matrix — AI Image Detector (Best F1={best_f1:.4f})')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout(); plt.savefig('/tmp/image_detector_cm.png', dpi=120)
plt.show()

from sklearn.metrics import roc_auc_score
auc = roc_auc_score(all_labels, all_probs)
print(f"AUC-ROC: {auc:.4f}")

In [ ]:
# ── CELL 11: Push to HuggingFace Hub ─────────────────────────
from huggingface_hub import HfApi
import json, os

model.push_to_hub(HF_REPO, commit_message=f"Fine-tuned Swin — F1={best_f1:.4f}")

# Also save locally + push feature extractor
from transformers import AutoFeatureExtractor
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
feature_extractor.push_to_hub(HF_REPO)

# Push model card
card = f"""---
license: apache-2.0
tags:
  - image-classification
  - ai-detection
  - deepfake-detection
  - diffusion-detection
datasets:
  - poloclub/diffusiondb
  - judegrant/cifake
  - saghi776/detectai-dataset
model-index:
  - name: aiscern-image-detector
    results:
      - task:
          type: image-classification
        metrics:
          - type: f1
            value: {best_f1:.4f}
---

# DETECTAI — AI Image Detector

Detects AI-generated images vs real photographs.

## Targets
- Stable Diffusion / SDXL / Flux  
- Midjourney v5/v6  
- DALL-E 3 (ChatGPT)  
- Google Gemini Nano / Imagen  
- Grok Aurora  
- Adobe Firefly / Canva AI  
- Real human faces, real photographs  

## Architecture
- Base: `microsoft/swin-base-patch4-window7-224`  
- Task: Binary image classification (0=real, 1=ai)  
- Fine-tuned on DiffusionDB + CIFAKE + platform data  

## Usage
```python
from transformers import pipeline
pipe = pipeline("image-classification", model="saghi776/aiscern-image-detector")
result = pipe("image.jpg")
```
"""

api = HfApi()
api.upload_file(
    path_or_fileobj=card.encode(),
    path_in_repo="README.md",
    repo_id=HF_REPO,
    repo_type="model",
    commit_message="Add model card"
)

print(f"\n✅ Model pushed to: https://huggingface.co/{HF_REPO}")
print(f"\n🔗 WEBSITE INTEGRATION:")
print(f"   File: frontend/lib/inference/hf-analyze.ts")
print(f"   Set:  MODELS.image_primary = '{HF_REPO}'")